In [ ]:
!pip uninstall -y torch torchaudio torchvision

!pip install torch==2.2.2 torchaudio==2.2.2 torchvision==0.17.2 --index-url https://download.pytorch.org/whl/cu121

Found existing installation: torch 2.2.2+cu121
Uninstalling torch-2.2.2+cu121:
  Successfully uninstalled torch-2.2.2+cu121
Found existing installation: torchaudio 2.2.2+cu121
Uninstalling torchaudio-2.2.2+cu121:
  Successfully uninstalled torchaudio-2.2.2+cu121
Found existing installation: torchvision 0.17.2+cu121
Uninstalling torchvision-0.17.2+cu121:
  Successfully uninstalled torchvision-0.17.2+cu121
Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.2.2%2Bcu121-cp312-cp312-linux_x86_64.whl (757.2 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchaudio-2.2.2%2Bcu121-cp312-cp312-linux_x86_64.whl (3.4 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchvision-0.17.2%2Bcu121-cp312-cp312-linux_x86_64.whl (7.0 MB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)


In [ ]:
!pip uninstall -y transformers
!pip install transformers datasets accelerate -q

Found existing installation: transformers 5.5.4
Uninstalling transformers-5.5.4:
  Successfully uninstalled transformers-5.5.4


In [ ]:
#Mount drive
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#Imports
import pandas as pd
import torch
from datasets import load_from_disk

In [ ]:
#Load sep28K labels
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/ml-stuttering-events-dataset/SEP-28k_labels.csv")

In [ ]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset

In [ ]:
#Define label groups
import numpy as np

TARGET_LABELS = [
    "Prolongation",
    "Block",
    "SoundRep",
    "WordRep",
    "Interjection"
]

def encode_labels(row):
    return np.array([
        1 if row[label] > 0 else 0
        for label in TARGET_LABELS
    ], dtype=np.float32)

df["labels"] = df.apply(encode_labels, axis=1)

In [ ]:
#Filter bad quality data
df = df[df["Unsure"] == 0]
df = df[df["PoorAudioQuality"] == 0]
df = df[df["DifficultToUnderstand"] == 0]
df = df[df["Music"] == 0]
df = df[df["NoSpeech"] == 0]

In [ ]:
#Full path
import os

BASE_DIR = "/content/drive/MyDrive/ml-stuttering-events-dataset"

df["full_path"] = df.apply(
    lambda row: os.path.join(
        BASE_DIR,
        f"sep28k/clips/{row['Show']}/{row['EpId']}/{row['Show']}_{row['EpId']}_{row['ClipId']}.wav"
    ),
    axis=1
)

In [ ]:
#Filter existing files
import os

df["exists"] = df["full_path"].apply(os.path.exists)
df = df[df["exists"] == True].reset_index(drop=True)

In [ ]:
#Convert to HF
from datasets import Dataset

dataset = Dataset.from_pandas(df)

NameError: name 'df' is not defined

In [ ]:
#Split again
dataset = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = dataset["train"]
val_dataset = dataset["test"]

In [ ]:
#Save correctly
train_dataset.save_to_disk("/content/drive/MyDrive/ml-stuttering-events-dataset/final_train")
val_dataset.save_to_disk("/content/drive/MyDrive/ml-stuttering-events-dataset/final_val")

Saving the dataset (0/1 shards):   0%|          | 0/14074 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1564 [00:00<?, ? examples/s]

In [ ]:
#Load saved dataset
from datasets import load_from_disk

train_dataset = load_from_disk("/content/drive/MyDrive/ml-stuttering-events-dataset/final_train")
val_dataset = load_from_disk("/content/drive/MyDrive/ml-stuttering-events-dataset/final_val")

In [ ]:
print(train_dataset.column_names)
print(train_dataset[0]["labels"])

['Show', 'EpId', 'ClipId', 'Start', 'Stop', 'Unsure', 'PoorAudioQuality', 'Prolongation', 'Block', 'SoundRep', 'WordRep', 'DifficultToUnderstand', 'Interjection', 'NoStutteredWords', 'NaturalPause', 'Music', 'NoSpeech', 'labels', 'full_path', 'exists']
[1.0, 0.0, 0.0, 1.0, 1.0]


In [ ]:
print(train_dataset[0])
print(type(train_dataset[0]["labels"]))
print(len(train_dataset[0]["labels"]))

{'Show': 'WomenWhoStutter', 'EpId': 25, 'ClipId': 4, 'Start': 4446560, 'Stop': 4494560, 'Unsure': 0, 'PoorAudioQuality': 0, 'Prolongation': 1, 'Block': 0, 'SoundRep': 0, 'WordRep': 1, 'DifficultToUnderstand': 0, 'Interjection': 2, 'NoStutteredWords': 2, 'NaturalPause': 1, 'Music': 0, 'NoSpeech': 0, 'labels': [1.0, 0.0, 0.0, 1.0, 1.0], 'full_path': '/content/drive/MyDrive/ml-stuttering-events-dataset/sep28k/clips/WomenWhoStutter/25/WomenWhoStutter_25_4.wav', 'exists': True}
<class 'list'>
5


In [ ]:
#Merge lables with saved dataset
from datasets import load_from_disk

train_dataset = load_from_disk("/content/drive/MyDrive/ml-stuttering-events-dataset/final_train")
val_dataset = load_from_disk("/content/drive/MyDrive/ml-stuttering-events-dataset/final_val")

In [ ]:
#Add labels back
def add_labels(example, idx):
    example["labels"] = df.iloc[idx]["labels"]
    return example

train_dataset = train_dataset.map(add_labels, with_indices=True)
val_dataset = val_dataset.map(add_labels, with_indices=True)

In [ ]:
#filter valid files
import os

df["exists"] = df["full_path"].apply(os.path.exists)

print(df["exists"].value_counts())

NameError: name 'df' is not defined

In [ ]:
#keep only valid files
df = df[df["exists"] == True].reset_index(drop=True)

In [ ]:
#Check valid data
print(len(df))
print(df["exists"].value_counts())

15638
exists
True    15638
Name: count, dtype: int64


In [ ]:
train_dataset = train_dataset.map(preprocess)
val_dataset = val_dataset.map(preprocess)

Map:   0%|          | 0/14074 [00:00<?, ? examples/s]

Map:   0%|          | 0/1564 [00:00<?, ? examples/s]

In [ ]:
#Save cleaned df
df.to_csv("/content/drive/MyDrive/ml-stuttering-events-dataset/clean_df.csv", index=False)

In [ ]:
#Model setup
from transformers import Wav2Vec2ForSequenceClassification

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "facebook/wav2vec2-xls-r-300m",
    num_labels=5,
    problem_type="multi_label_classification",
    ignore_mismatched_sizes=True
)

ImportError: 
Wav2Vec2ForSequenceClassification requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.


In [ ]:
#Override loss inside trainer
import torch.nn as nn
from transformers import Trainer

class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):  # 👈 add **kwargs
        labels = inputs.get("labels")

        outputs = model(input_values=inputs["input_values"])
        logits = outputs.logits

        loss_fct = nn.BCEWithLogitsLoss()
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [ ]:
#Checkoutput
print(df[["labels"]].head())

NameError: name 'df' is not defined

In [ ]:
#Load dataset first time
from datasets import Dataset

dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.1)

train_dataset = dataset["train"]
val_dataset = dataset["test"]

In [ ]:
#Load dataset after saving
from datasets import load_from_disk

train_dataset = load_from_disk("/content/drive/MyDrive/ml-stuttering-events-dataset/final_train")
val_dataset = load_from_disk("/content/drive/MyDrive/ml-stuttering-events-dataset/final_val")

FileNotFoundError: Directory /content/drive/MyDrive/ml-stuttering-events-dataset/final_train not found

In [ ]:
print(train_dataset.column_names)

['Show', 'EpId', 'ClipId', 'Start', 'Stop', 'Unsure', 'PoorAudioQuality', 'Prolongation', 'Block', 'SoundRep', 'WordRep', 'DifficultToUnderstand', 'Interjection', 'NoStutteredWords', 'NaturalPause', 'Music', 'NoSpeech', 'full_path', '__index_level_0__']


In [ ]:
#Feature extractor
import torchaudio
import numpy as np
from transformers import Wav2Vec2FeatureExtractor

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
    "facebook/wav2vec2-xls-r-300m"
)

In [ ]:
print(train_dataset.column_names)

['Show', 'EpId', 'ClipId', 'Start', 'Stop', 'Unsure', 'PoorAudioQuality', 'Prolongation', 'Block', 'SoundRep', 'WordRep', 'DifficultToUnderstand', 'Interjection', 'NoStutteredWords', 'NaturalPause', 'Music', 'NoSpeech', 'labels', 'full_path', 'exists']


In [ ]:
#Create input values
def preprocess(example):
    waveform, sr = torchaudio.load(example["full_path"])

    # convert to mono if stereo
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    # resample to 16k
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(sr, 16000)
        waveform = resampler(waveform)

    inputs = feature_extractor(
        waveform.squeeze().numpy(),
        sampling_rate=16000
    )

    example["input_values"] = inputs["input_values"][0]

    return example

In [ ]:
#Apply it
train_dataset = train_dataset.map(preprocess)
val_dataset = val_dataset.map(preprocess)

Map:   0%|          | 0/14074 [00:00<?, ? examples/s]

Map:   0%|          | 0/1564 [00:00<?, ? examples/s]

In [ ]:
#Save the dataset after preprocessing
train_dataset.save_to_disk("/content/drive/MyDrive/train_ready")
val_dataset.save_to_disk("/content/drive/MyDrive/val_ready")

NameError: name 'train_dataset' is not defined

In [ ]:
print(train_dataset[0].keys())

dict_keys(['Show', 'EpId', 'ClipId', 'Start', 'Stop', 'Unsure', 'PoorAudioQuality', 'Prolongation', 'Block', 'SoundRep', 'WordRep', 'DifficultToUnderstand', 'Interjection', 'NoStutteredWords', 'NaturalPause', 'Music', 'NoSpeech', 'labels', 'full_path', 'exists', 'input_values'])


In [ ]:
#Data collator
import torch.nn.utils.rnn as rnn_utils

def collate_fn(batch):
    input_values = [
        torch.tensor(x["input_values"], dtype=torch.float32)
        for x in batch
    ]

    labels = [
        torch.tensor(x["labels"], dtype=torch.float32)
        for x in batch
    ]

    input_values = rnn_utils.pad_sequence(input_values, batch_first=True)

    return {
        "input_values": input_values,
        "labels": torch.stack(labels)
    }

In [ ]:
#Training arguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./stutter_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    logging_steps=10,
    report_to="none"
)

In [ ]:
#Define metrics
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs > 0.5).astype(int)

    return {
        "f1_micro": f1_score(labels, preds, average="micro"),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "accuracy": accuracy_score(labels, preds)
    }

In [ ]:
#Initialize Trainer
from transformers import Trainer

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
    compute_metrics=compute_metrics
)

NameError: name 'model' is not defined

In [ ]:
import numpy as np

In [ ]:
#Start training
trainer.train()

NameError: name 'trainer' is not defined

In [ ]:
trainer.evaluate()

{'eval_loss': 0.41292649507522583,
 'eval_f1_micro': 0.6598274656096992,
 'eval_f1_macro': 0.6604195435434804,
 'eval_accuracy': 0.36381074168797956,
 'eval_runtime': 52.3483,
 'eval_samples_per_second': 29.877,
 'eval_steps_per_second': 7.469,
 'epoch': 5.0}

In [ ]:
predictions = trainer.predict(val_dataset)

NameError: name 'trainer' is not defined

In [ ]:
#Extract values
logits = predictions.predictions
labels = predictions.label_ids

NameError: name 'predictions' is not defined

In [ ]:
from sklearn.metrics import classification_report

TARGET_LABELS = [
    "Prolongation",
    "Block",
    "SoundRep",
    "WordRep",
    "Interjection"
]

report = classification_report(
    labels,
    preds,
    target_names=TARGET_LABELS,
    output_dict=True
)

for label in TARGET_LABELS:
    print(f"{label}: F1 = {report[label]['f1-score']:.3f}")

Prolongation: F1 = 0.602
Block: F1 = 0.547
SoundRep: F1 = 0.631
WordRep: F1 = 0.705
Interjection: F1 = 0.817


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
#Save the model
trainer.save_model("/content/drive/MyDrive/stutter_model_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
#Save training arg and setup (For documentation)
import json

with open("/content/drive/MyDrive/stutter_model_final/training_info.json", "w") as f:
    json.dump(training_args.to_dict(), f, indent=4)


In [ ]:
#Save eval results (For documentation)
results = trainer.evaluate()

import json

with open("/content/drive/MyDrive/stutter_model_final/eval_results.json", "w") as f:
    json.dump(results, f, indent=4)

In [ ]:
#IMPROVING FIRST SCORES - changed threshold no retraining

In [ ]:
#Load imports/model/dataset
import torch
import numpy as np
from datasets import load_from_disk
from transformers import Wav2Vec2ForSequenceClassification
from transformers import Trainer
from sklearn.metrics import classification_report
import torch.nn.utils.rnn as rnn_utils

train_dataset = load_from_disk("/content/drive/MyDrive/train_ready")
val_dataset = load_from_disk("/content/drive/MyDrive/val_ready")

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "/content/drive/MyDrive/stutter_model_final"
)

ImportError: 
Wav2Vec2ForSequenceClassification requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.


In [ ]:
#Save labels separatly (To stop cross entropy)
labels = np.array([x["labels"] for x in val_dataset])

#Remove lables from dataset
val_dataset = val_dataset.remove_columns("labels")

In [ ]:
#Recreate collator (No labels)
import torch.nn.utils.rnn as rnn_utils

def collate_fn(batch):
    input_values = [
        torch.tensor(x["input_values"], dtype=torch.float32)
        for x in batch
    ]

    input_values = rnn_utils.pad_sequence(input_values, batch_first=True)

    return {
        "input_values": input_values
    }

In [ ]:
#Recreate trainer (No args only for predict)
trainer = Trainer(
    model=model,
    data_collator=collate_fn
)

In [ ]:
#Re-evaluate
from sklearn.metrics import classification_report

predictions = trainer.predict(val_dataset)

logits = predictions.predictions

#Threshold tuning
probs = 1 / (1 + np.exp(-logits))
preds = (probs > 0.3).astype(int)

#Report
TARGET_LABELS = [
    "Prolongation",
    "Block",
    "SoundRep",
    "WordRep",
    "Interjection"
]

report = classification_report(labels, preds, target_names=TARGET_LABELS, output_dict=True)

for label in TARGET_LABELS:
    print(f"{label}: F1 = {report[label]['f1-score']:.3f}")

Prolongation: F1 = 0.623
Block: F1 = 0.649
SoundRep: F1 = 0.621
WordRep: F1 = 0.738
Interjection: F1 = 0.819


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
#IMPROVE SCORES - With retraining

In [ ]:
#Imports
import torch
import numpy as np
from datasets import load_from_disk
from transformers import (
    Wav2Vec2ForSequenceClassification,
    TrainingArguments,
    Trainer
)
import torch.nn as nn
import torch.nn.utils.rnn as rnn_utils

In [ ]:
#Load dataset
train_dataset = load_from_disk("/content/drive/MyDrive/train_ready")
val_dataset = load_from_disk("/content/drive/MyDrive/val_ready")

In [ ]:
#Load model
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "/content/drive/MyDrive/stutter_model_final",
    ignore_mismatched_sizes=True
)

ImportError: 
Wav2Vec2ForSequenceClassification requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.


In [ ]:
#Unfreeze for improvement
for param in model.wav2vec2.feature_extractor.parameters():
    param.requires_grad = False

for param in model.wav2vec2.encoder.layers[-6:]:
    for p in param.parameters():
        p.requires_grad = True

In [ ]:
#Trainer - Stronger loss (Big upgrade)
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs["labels"]

        outputs = model(input_values=inputs["input_values"])
        logits = outputs.logits

        #Stronger focus on weak classes (Like block)
        pos_weights = torch.tensor([1.3, 1.9, 1.2, 1.0, 1.4]).to(logits.device)

        loss_fct = nn.BCEWithLogitsLoss(pos_weight=pos_weights)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [ ]:
#Trainer args
training_args = TrainingArguments(
    output_dir="./stutter_model_v2",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=8,   #Better than 5
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=10,
    report_to="none"
)

In [ ]:
#Collator
def collate_fn(batch):
    input_values = [
        torch.tensor(x["input_values"], dtype=torch.float32)
        for x in batch
    ]

    labels = [
        torch.tensor(x["labels"], dtype=torch.float32)
        for x in batch
    ]

    input_values = rnn_utils.pad_sequence(input_values, batch_first=True)

    return {
        "input_values": input_values,
        "labels": torch.stack(labels)
    }

In [ ]:
#Create trainer
trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.430993,0.516498
2,0.502197,0.500818
3,0.408900,0.522516
4,0.418519,0.515055
5,0.418857,0.531380
6,0.234788,0.554195
7,0.381400,0.556953
8,0.340939,0.568812


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=28152, training_loss=0.39695857411065744, metrics={'train_runtime': 6103.1992, 'train_samples_per_second': 18.448, 'train_steps_per_second': 4.613, 'total_flos': 1.0237122772922878e+19, 'train_loss': 0.39695857411065744, 'epoch': 8.0})

In [ ]:
#Save new model
trainer.save_model("/content/drive/MyDrive/stutter_model_v2")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
#Evaluate (Threshold 0.3)
predictions = trainer.predict(val_dataset)

logits = predictions.predictions
labels = predictions.label_ids

probs = 1 / (1 + np.exp(-logits))
preds = (probs > 0.3).astype(int)

from sklearn.metrics import classification_report

TARGET_LABELS = [
    "Prolongation",
    "Block",
    "SoundRep",
    "WordRep",
    "Interjection"
]

print(classification_report(labels, preds, target_names=TARGET_LABELS))

              precision    recall  f1-score   support

Prolongation       0.52      0.71      0.60       449
       Block       0.54      0.86      0.66       664
    SoundRep       0.56      0.69      0.62       295
     WordRep       0.76      0.69      0.73       306
Interjection       0.81      0.80      0.81       614

   micro avg       0.62      0.77      0.68      2328
   macro avg       0.64      0.75      0.68      2328
weighted avg       0.64      0.77      0.69      2328
 samples avg       0.53      0.62      0.55      2328



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
#Improve the model again with no training

In [ ]:
#Imports
import numpy as np
from datasets import load_from_disk
from transformers import Trainer, Wav2Vec2ForSequenceClassification
from sklearn.metrics import classification_report

In [ ]:
#Load dataset and model
val_dataset = load_from_disk("/content/drive/MyDrive/val_ready")

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "/content/drive/MyDrive/stutter_model_final"
)

ImportError: 
Wav2Vec2ForSequenceClassification requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.


In [ ]:
#Copy with no labels
val_no_labels = val_dataset.remove_columns(["labels"])

#Save labels separatly
labels = np.array(val_dataset["labels"])

In [ ]:
#Create trainer (Inference only)
trainer = Trainer(
    model=model)

In [ ]:
#Get raw predictions
predictions = trainer.predict(val_no_labels)
logits = predictions.predictions

probs = 1 / (1 + np.exp(-logits))

#Global threshold search (Find best one)
from sklearn.metrics import classification_report

best_t = 0
best_f1 = 0

for t in np.arange(0.20, 0.50, 0.05):
    preds = (probs > t).astype(int)

    report = classification_report(
        labels,
        preds,
        output_dict=True,
        zero_division=0
    )

    f1 = report["macro avg"]["f1-score"]

    print(f"Threshold {t:.2f} → F1: {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print("\nBEST GLOBAL THRESHOLD:", best_t)
print("BEST F1:", best_f1)

Threshold 0.20 → F1: 0.6853
Threshold 0.25 → F1: 0.6925
Threshold 0.30 → F1: 0.6900
Threshold 0.35 → F1: 0.6883
Threshold 0.40 → F1: 0.6798
Threshold 0.45 → F1: 0.6683

BEST GLOBAL THRESHOLD: 0.25
BEST F1: 0.6924875147551065


In [ ]:
#Apply best one
global_preds = (probs > best_t).astype(int)

In [ ]:
#See results
TARGET_LABELS = [
    "Prolongation",
    "Block",
    "SoundRep",
    "WordRep",
    "Interjection"
]

print(classification_report(labels, global_preds, target_names=TARGET_LABELS))

              precision    recall  f1-score   support

Prolongation       0.52      0.79      0.63       449
       Block       0.53      0.85      0.66       664
    SoundRep       0.54      0.73      0.62       295
     WordRep       0.77      0.71      0.74       306
Interjection       0.82      0.81      0.82       614

   micro avg       0.61      0.79      0.69      2328
   macro avg       0.64      0.78      0.69      2328
weighted avg       0.64      0.79      0.70      2328
 samples avg       0.53      0.64      0.55      2328



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
#Per-label threshold optimization
from sklearn.metrics import f1_score

thresholds = [0.2, 0.25, 0.3, 0.35, 0.4, 0.45]

best_thresholds = []

for i in range(5):
    best_t = 0
    best_f1 = 0

    for t in thresholds:
        preds = (probs[:, i] > t).astype(int)

        f1 = f1_score(labels[:, i], preds)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    best_thresholds.append(best_t)
    print(f"{TARGET_LABELS[i]} → best threshold: {best_t} | F1: {best_f1:.3f}")

Prolongation → best threshold: 0.25 | F1: 0.628
Block → best threshold: 0.2 | F1: 0.664
SoundRep → best threshold: 0.4 | F1: 0.641
WordRep → best threshold: 0.3 | F1: 0.738
Interjection → best threshold: 0.3 | F1: 0.819


In [ ]:
#Final predictions
final_preds = np.zeros_like(probs)

for i in range(5):
    final_preds[:, i] = (probs[:, i] > best_thresholds[i]).astype(int)

print(classification_report(labels, final_preds, target_names=TARGET_LABELS))

              precision    recall  f1-score   support

Prolongation       0.52      0.79      0.63       449
       Block       0.52      0.92      0.66       664
    SoundRep       0.63      0.65      0.64       295
     WordRep       0.79      0.69      0.74       306
Interjection       0.85      0.79      0.82       614

   micro avg       0.62      0.80      0.70      2328
   macro avg       0.66      0.77      0.70      2328
weighted avg       0.66      0.80      0.70      2328
 samples avg       0.54      0.64      0.56      2328



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
#New training (weighted loss/multi-label setup/stable training)

In [ ]:
import torch
import numpy as np
from datasets import load_from_disk
from transformers import (
    AutoFeatureExtractor,
    AutoModelForAudioClassification,
    TrainingArguments,
    Trainer
)

In [ ]:
#Mount drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
train_dataset = load_from_disk("/content/drive/MyDrive/train_ready")
val_dataset = load_from_disk("/content/drive/MyDrive/val_ready")

In [ ]:
#Check dataset for issues
print(train_dataset[0].keys())
print(type(train_dataset[0]["labels"]))

dict_keys(['Show', 'EpId', 'ClipId', 'Start', 'Stop', 'Unsure', 'PoorAudioQuality', 'Prolongation', 'Block', 'SoundRep', 'WordRep', 'DifficultToUnderstand', 'Interjection', 'NoStutteredWords', 'NaturalPause', 'Music', 'NoSpeech', 'labels', 'full_path', 'exists', 'input_values'])
<class 'list'>


In [ ]:
print(type(train_dataset[0]["labels"]))
print(len(train_dataset[0]["labels"]))

<class 'list'>
5


In [ ]:
#Feature extractor
from transformers import AutoFeatureExtractor

feature_extractor = AutoFeatureExtractor.from_pretrained(
    "facebook/wav2vec2-xls-r-300m"
)

In [ ]:
#Class weights
import numpy as np
import torch

labels_array = np.array(train_dataset["labels"])

class_counts = labels_array.sum(axis=0)
weights = class_counts.max() / (class_counts + 1e-6)

weights = torch.tensor(weights, dtype=torch.float32)

print(weights)

tensor([1.3097, 1.0000, 2.1307, 2.0726, 1.0674])


In [ ]:
#Model
import torch
import torch.nn as nn
from transformers import AutoModel

base = AutoModel.from_pretrained(
    "facebook/wav2vec2-xls-r-300m"
)

class StutterModel(nn.Module):
    def __init__(self, base, weights):
        super().__init__()
        self.base = base
        self.classifier = nn.Linear(1024, 5)  # FIXED
        self.weights = weights

    def forward(self, input_values, labels=None):
        outputs = self.base(input_values)

        x = outputs.last_hidden_state.mean(dim=1)

        logits = self.classifier(x)

        loss = None
        if labels is not None:
            loss_fn = nn.BCEWithLogitsLoss(
                pos_weight=self.weights.to(logits.device)
            )
            loss = loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}

model = StutterModel(base, weights)

ImportError: 
AutoModel requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.


In [ ]:
#Collator
import torch
import numpy as np

def collate_fn(batch):
    input_values = [
        torch.tensor(x["input_values"], dtype=torch.float32)
        for x in batch
    ]

    labels = torch.tensor(
        np.array([x["labels"] for x in batch]),
        dtype=torch.float32
    )

    input_values = torch.nn.utils.rnn.pad_sequence(
        input_values,
        batch_first=True
    )

    return {
        "input_values": input_values,
        "labels": labels
    }

In [ ]:
#Training args
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./stutter_model",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=8,
    learning_rate=1e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    fp16=True
)

In [ ]:
#Metrics
from sklearn.metrics import f1_score, precision_score, recall_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs > 0.5).astype(int)

    return {
        "f1": f1_score(labels, preds, average="micro"),
        "precision": precision_score(labels, preds, average="micro"),
        "recall": recall_score(labels, preds, average="micro"),
    }

In [ ]:
#Trainer
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
)

NameError: name 'model' is not defined

In [ ]:
#train
trainer.train()

Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.621366,0.602891,0.521315,0.648148,0.435997
2,0.564960,0.556154,0.585010,0.705704,0.499570
3,0.508808,0.535802,0.612389,0.744315,0.520189
4,0.540287,0.513601,0.639849,0.712329,0.580756


Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.621366,0.602891,0.521315,0.648148,0.435997
2,0.564960,0.556154,0.585010,0.705704,0.499570
3,0.508808,0.535802,0.612389,0.744315,0.520189
4,0.540287,0.513601,0.639849,0.712329,0.580756
5,0.494014,0.511745,0.645068,0.737832,0.573024
6,0.481959,0.511969,0.652164,0.735310,0.585911
7,0.485382,0.501831,0.659579,0.732808,0.599656
8,0.432914,0.505266,0.656168,0.738057,0.590636


TrainOutput(global_step=28152, training_loss=0.5340846147717451, metrics={'train_runtime': 5838.1489, 'train_samples_per_second': 19.286, 'train_steps_per_second': 4.822, 'total_flos': 0.0, 'train_loss': 0.5340846147717451, 'epoch': 8.0})

In [ ]:
#Get predictions
preds = trainer.predict(val_dataset)

logits = preds.predictions
labels = preds.label_ids

#Convert to probabilities
import numpy as np

probs = 1 / (1 + np.exp(-logits))

#Apply best thresholds
thresholds = np.array([0.25, 0.20, 0.40, 0.30, 0.30])

predictions = (probs > thresholds).astype(int)

NameError: name 'trainer' is not defined

In [ ]:
#Results
from sklearn.metrics import classification_report

print(classification_report(labels, predictions, target_names=[
    "Prolongation",
    "Block",
    "SoundRep",
    "WordRep",
    "Interjection"
]))

              precision    recall  f1-score   support

Prolongation       0.52      0.78      0.63       449
       Block       0.50      0.93      0.65       664
    SoundRep       0.61      0.69      0.65       295
     WordRep       0.76      0.66      0.70       306
Interjection       0.81      0.81      0.81       614

   micro avg       0.60      0.80      0.69      2328
   macro avg       0.64      0.77      0.69      2328
weighted avg       0.63      0.80      0.69      2328
 samples avg       0.51      0.64      0.54      2328



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
#Recompute thresholds
import numpy as np
from sklearn.metrics import f1_score

class_names = [
    "Prolongation",
    "Block",
    "SoundRep",
    "WordRep",
    "Interjection"
]

best_thresholds = []

for i, name in enumerate(class_names):
    best_t = 0.5
    best_f1 = 0

    for t in np.arange(0.1, 0.9, 0.05):
        preds = (probs[:, i] > t).astype(int)
        f1 = f1_score(labels[:, i], preds)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    print(f"{name} → best threshold: {best_t} | F1: {best_f1:.3f}")
    best_thresholds.append(best_t)

print("Final thresholds:", best_thresholds)

Prolongation → best threshold: 0.3500000000000001 | F1: 0.631
Block → best threshold: 0.25000000000000006 | F1: 0.656
SoundRep → best threshold: 0.40000000000000013 | F1: 0.647
WordRep → best threshold: 0.15000000000000002 | F1: 0.710
Interjection → best threshold: 0.3500000000000001 | F1: 0.822
Final thresholds: [np.float64(0.3500000000000001), np.float64(0.25000000000000006), np.float64(0.40000000000000013), np.float64(0.15000000000000002), np.float64(0.3500000000000001)]


In [ ]:
#Use them
thresholds = np.array(best_thresholds)
predictions = (probs > thresholds).astype(int)

In [ ]:
#Results with new thresholds
from sklearn.metrics import classification_report

print(classification_report(labels, predictions, target_names=[
    "Prolongation",
    "Block",
    "SoundRep",
    "WordRep",
    "Interjection"
]))

              precision    recall  f1-score   support

Prolongation       0.60      0.67      0.63       449
       Block       0.53      0.85      0.66       664
    SoundRep       0.61      0.69      0.65       295
     WordRep       0.68      0.75      0.71       306
Interjection       0.85      0.80      0.82       614

   micro avg       0.64      0.77      0.70      2328
   macro avg       0.65      0.75      0.69      2328
weighted avg       0.66      0.77      0.70      2328
 samples avg       0.52      0.61      0.54      2328



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
#Save the model
trainer.save_model("/content/drive/MyDrive/stutter_model_v3")

In [ ]:
#Save threholds
import numpy as np

np.save("/content/drive/MyDrive/best_thresholds.npy", thresholds)

#Save predictions
np.save("/content/drive/MyDrive/val_probs.npy", probs)
np.save("/content/drive/MyDrive/val_labels.npy", labels)

#Save trainer state
trainer.save_state()

In [ ]:
#New training

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA A100-SXM4-40GB


In [ ]:
import torch
print(torch.__version__)

2.10.0+cu128


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn as nn
import numpy as np
from datasets import load_from_disk
from transformers import AutoModel, Trainer
import torch.nn.utils.rnn as rnn_utils
from sklearn.metrics import classification_report

In [ ]:
train_dataset = load_from_disk("/content/drive/MyDrive/train_ready")
val_dataset = load_from_disk("/content/drive/MyDrive/val_ready")

In [ ]:
labels_array = np.array(train_dataset["labels"])
class_counts = labels_array.sum(axis=0)

weights = class_counts.max() / (class_counts + 1e-6)
weights = torch.tensor(weights, dtype=torch.float32)

In [ ]:
from transformers import Wav2Vec2Model
import torch
import torch.nn as nn

base = Wav2Vec2Model.from_pretrained(
    "facebook/wav2vec2-xls-r-300m"
)

class StutterModelV4(nn.Module):
    def __init__(self, base, weights):
        super().__init__()
        self.base = base
        hidden = base.config.hidden_size

        self.attention = nn.Linear(hidden, 1)

        self.classifier = nn.Sequential(
            nn.Linear(hidden, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 5)
        )

        self.weights = weights

    def forward(self, input_values, labels=None):
        outputs = self.base(input_values)

        x = outputs.last_hidden_state

        attn_scores = self.attention(x)
        attn_weights = torch.softmax(attn_scores, dim=1)
        x = (x * attn_weights).sum(dim=1)

        logits = self.classifier(x)

        loss = None
        if labels is not None:
            loss_fn = nn.BCEWithLogitsLoss(
                pos_weight=self.weights.to(logits.device)
            )
            loss = loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.bias             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
model = StutterModelV4(base, weights)

In [ ]:
def collate_fn(batch):
    input_values = [
        torch.tensor(x["input_values"], dtype=torch.float32)
        for x in batch
    ]

    input_values = rnn_utils.pad_sequence(input_values, batch_first=True)

    labels = torch.tensor(
        np.array([x["labels"] for x in batch]),
        dtype=torch.float32
    )

    return {
        "input_values": input_values,
        "labels": labels
    }

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./stutter_v4",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=8,
    weight_decay=0.01,
    logging_steps=10,
    report_to="none"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.469905,0.513149
2,0.490067,0.501615
3,0.406252,0.511937
4,0.391369,0.501520
5,0.420839,0.511288
6,0.379574,0.519234
7,0.484487,0.516726
8,0.422935,0.518837


TrainOutput(global_step=28152, training_loss=0.45054765838338895, metrics={'train_runtime': 6229.589, 'train_samples_per_second': 18.074, 'train_steps_per_second': 4.519, 'total_flos': 0.0, 'train_loss': 0.45054765838338895, 'epoch': 8.0})

In [ ]:
trainer.save_model("/content/drive/MyDrive/stutter_v4_checkpoint")
trainer.save_state()

In [ ]:
preds = trainer.predict(val_dataset)

logits = preds.predictions
labels = preds.label_ids

In [ ]:
import numpy as np

probs = 1 / (1 + np.exp(-logits))

In [ ]:
thresholds = np.array([0.25, 0.20, 0.40, 0.30, 0.30])

predictions = (probs > thresholds).astype(int)

In [ ]:
from sklearn.metrics import classification_report

TARGET_LABELS = [
    "Prolongation",
    "Block",
    "SoundRep",
    "WordRep",
    "Interjection"
]

print(classification_report(labels, predictions, target_names=TARGET_LABELS))

              precision    recall  f1-score   support

Prolongation       0.53      0.75      0.62       449
       Block       0.53      0.89      0.66       664
    SoundRep       0.59      0.69      0.64       295
     WordRep       0.78      0.73      0.75       306
Interjection       0.83      0.81      0.82       614

   micro avg       0.62      0.80      0.70      2328
   macro avg       0.65      0.77      0.70      2328
weighted avg       0.65      0.80      0.70      2328
 samples avg       0.53      0.63      0.55      2328



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from sklearn.metrics import f1_score

best_thresholds = []

for i, name in enumerate(TARGET_LABELS):
    best_t = 0.5
    best_f1 = 0

    for t in np.arange(0.1, 0.9, 0.05):
        pred_i = (probs[:, i] > t).astype(int)
        f1 = f1_score(labels[:, i], pred_i)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    print(f"{name} → best threshold: {best_t:.2f} | F1: {best_f1:.3f}")
    best_thresholds.append(best_t)

Prolongation → best threshold: 0.30 | F1: 0.637
Block → best threshold: 0.20 | F1: 0.663
SoundRep → best threshold: 0.50 | F1: 0.649
WordRep → best threshold: 0.40 | F1: 0.760
Interjection → best threshold: 0.25 | F1: 0.820


In [ ]:
best_thresholds = np.array(best_thresholds)

final_predictions = (probs > best_thresholds).astype(int)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    labels,
    final_predictions,
    target_names=[
        "Prolongation",
        "Block",
        "SoundRep",
        "WordRep",
        "Interjection"
    ]
))

              precision    recall  f1-score   support

Prolongation       0.57      0.72      0.64       449
       Block       0.53      0.89      0.66       664
    SoundRep       0.63      0.66      0.65       295
     WordRep       0.81      0.71      0.76       306
Interjection       0.82      0.82      0.82       614

   micro avg       0.64      0.79      0.70      2328
   macro avg       0.67      0.76      0.71      2328
weighted avg       0.66      0.79      0.71      2328
 samples avg       0.54      0.63      0.56      2328



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# Save the model
trainer.save_model("/content/drive/MyDrive/stutter_model_v4")

# Save best thresholds
import numpy as np
np.save("/content/drive/MyDrive/v4_best_thresholds.npy", best_thresholds)

# Save probabilities
np.save("/content/drive/MyDrive/v4_val_probs.npy", probs)

# Save labels
np.save("/content/drive/MyDrive/v4_val_labels.npy", labels)

# Save trainer state
trainer.save_state()

In [ ]:
#Connect Model to App

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import numpy as np
from transformers import Wav2Vec2Model

In [ ]:
#Model class
class StutterModelV4(torch.nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        hidden = base.config.hidden_size

        self.attention = torch.nn.Linear(hidden, 1)

        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(hidden, 512),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(512, 5)
        )

    def forward(self, input_values):
        outputs = self.base(input_values)
        x = outputs.last_hidden_state

        attn = torch.softmax(self.attention(x), dim=1)
        x = (x * attn).sum(dim=1)

        logits = self.classifier(x)
        return logits

In [ ]:
#Load base model
base = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-xls-r-300m")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
#Load trained weights
from safetensors.torch import load_file

model = StutterModelV4(base)

state_dict = load_file("/content/drive/MyDrive/stutter_model_v4/model.safetensors")
model.load_state_dict(state_dict)

model.eval()

StutterModelV4(
  (base): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection

In [ ]:
#Save inference ready model
torch.save(
    model.state_dict(),
    "/content/drive/MyDrive/stutter_v4_inference.pt"
)

In [ ]:
#Check thresholds
thresholds = np.load("/content/drive/MyDrive/v4_best_thresholds.npy")
print(thresholds)

[0.3  0.2  0.5  0.4  0.25]


In [ ]:
# Combine English and Arabic datasets for training the model on both languages
# Then test the model only on Arabic data to check performance

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Import libraries

import os
import torch
import torchaudio
import numpy as np
import pandas as pd

from datasets import Dataset, load_from_disk, concatenate_datasets
from transformers import AutoFeatureExtractor

In [ ]:
# Import pandas for data processing
import pandas as pd

# Path to Excel file
excel_path = "/content/drive/MyDrive/تأتأة (2).xlsx"

# Read Excel file (no header)
df_raw = pd.read_excel(excel_path, sheet_name=1, header=None)

In [ ]:
# Remove empty rows and reset index
df = df_raw.iloc[4:].reset_index(drop=True)

# Select ClipId and label columns only
df = df.iloc[:, [2, 3]]

# Rename columns
df.columns = ["ClipId", "label_text"]

# Remove empty rows
df = df.dropna()

In [ ]:
# Convert ClipId to numbers
df["ClipId"] = pd.to_numeric(df["ClipId"], errors="coerce")

# Remove invalid rows
df = df.dropna(subset=["ClipId"])

# Convert to integer
df["ClipId"] = df["ClipId"].astype(int)

# Create string version for matching audio files
df["ClipId_str"] = df["ClipId"].astype(str)

In [ ]:
# Import OS for file handling
import os

# Audio folder path
audio_folder = "/content/drive/MyDrive/Audio_All"

# Get all audio files
audio_files = os.listdir(audio_folder)

# Map file name (without .wav) → full path
audio_map = {}

for f in audio_files:
    name = f.replace(".wav", "")
    if name.isdigit():
        audio_map[name] = os.path.join(audio_folder, f)

In [ ]:
# Match ClipId with audio path
df["audio_path"] = df["ClipId_str"].map(audio_map)

# Remove rows without audio file
df = df.dropna(subset=["audio_path"])

In [ ]:
# Show raw data structure
print(df_raw.head())

# Show columns
print(df_raw.columns)

# Show final data
print(df.head())

    0   1           2                 3
0 NaN NaN         NaN               NaN
1 NaN NaN         NaN               NaN
2 NaN NaN         NaN               NaN
3 NaN NaN  رقم الفويس      نمط التاتاة 
4 NaN NaN           2  NoStutteredWords
Index([0, 1, 2, 3], dtype='int64')
   ClipId        label_text ClipId_str                              audio_path
0       2  NoStutteredWords          2  /content/drive/MyDrive/Audio_All/2.wav
1       3   Block, SoundRep          3  /content/drive/MyDrive/Audio_All/3.wav
2       4   Block, SoundRep          4  /content/drive/MyDrive/Audio_All/4.wav
3       5             Block          5  /content/drive/MyDrive/Audio_All/5.wav
4       6  NoStutteredWords          6  /content/drive/MyDrive/Audio_All/6.wav


In [ ]:
# Create empty binary label columns (all values start as 0)
df["Prolongation"] = 0
df["Block"] = 0
df["SoundRep"] = 0
df["WordRep"] = 0
df["Interjection"] = 0
df["NoStutteredWords"] = 0

In [ ]:
# This function reads the label text and assigns 1 if the pattern exists
def fill_labels(row):
    text = str(row["label_text"]).lower()

    # Check each stuttering type inside the text
    if "prolongation" in text:
        row["Prolongation"] = 1

    if "block" in text:
        row["Block"] = 1

    if "soundrep" in text or "sound rep" in text:
        row["SoundRep"] = 1

    if "wordrep" in text or "word rep" in text:
        row["WordRep"] = 1

    if "interjection" in text:
        row["Interjection"] = 1

    if "nostutteredwords" in text:
        row["NoStutteredWords"] = 1

    return row

In [ ]:
# Apply the function to every row in the dataframe
df = df.apply(fill_labels, axis=1)

In [ ]:
# Display first rows to verify labels are correct
print(df.head())

   ClipId        label_text ClipId_str  \
0       2  NoStutteredWords          2   
1       3   Block, SoundRep          3   
2       4   Block, SoundRep          4   
3       5             Block          5   
4       6  NoStutteredWords          6   

                               audio_path  Prolongation  Block  SoundRep  \
0  /content/drive/MyDrive/Audio_All/2.wav             0      0         0   
1  /content/drive/MyDrive/Audio_All/3.wav             0      1         1   
2  /content/drive/MyDrive/Audio_All/4.wav             0      1         1   
3  /content/drive/MyDrive/Audio_All/5.wav             0      1         0   
4  /content/drive/MyDrive/Audio_All/6.wav             0      0         0   

   WordRep  Interjection  NoStutteredWords  
0        0             0                 1  
1        0             0                 0  
2        0             0                 0  
3        0             0                 0  
4        0             0                 1  


In [ ]:
# Create final labels array

TARGET_LABELS = [
    "Prolongation",
    "Block",
    "SoundRep",
    "WordRep",
    "Interjection"
]

def encode_labels(row):

    return np.array([
        row[label]
        for label in TARGET_LABELS
    ], dtype=np.float32)

# Add labels column

df["labels"] = df.apply(
    encode_labels,
    axis=1
)

print(df["labels"].head())

0    [0.0, 0.0, 0.0, 0.0, 0.0]
1    [0.0, 1.0, 1.0, 0.0, 0.0]
2    [0.0, 1.0, 1.0, 0.0, 0.0]
3    [0.0, 1.0, 0.0, 0.0, 0.0]
4    [0.0, 0.0, 0.0, 0.0, 0.0]
Name: labels, dtype: object


In [ ]:
# Create dataset format

df["full_path"] = df["audio_path"]

arabic_df = df[[
    "full_path",
    "labels"
]].copy()

print(arabic_df.head())

                                full_path                     labels
0  /content/drive/MyDrive/Audio_All/2.wav  [0.0, 0.0, 0.0, 0.0, 0.0]
1  /content/drive/MyDrive/Audio_All/3.wav  [0.0, 1.0, 1.0, 0.0, 0.0]
2  /content/drive/MyDrive/Audio_All/4.wav  [0.0, 1.0, 1.0, 0.0, 0.0]
3  /content/drive/MyDrive/Audio_All/5.wav  [0.0, 1.0, 0.0, 0.0, 0.0]
4  /content/drive/MyDrive/Audio_All/6.wav  [0.0, 0.0, 0.0, 0.0, 0.0]


In [ ]:
# Convert to HuggingFace dataset

arabic_dataset = Dataset.from_pandas(
    arabic_df
)

print(arabic_dataset)

Dataset({
    features: ['full_path', 'labels', '__index_level_0__'],
    num_rows: 118
})


In [ ]:
# Load feature extractor

feature_extractor = AutoFeatureExtractor.from_pretrained(
    "facebook/wav2vec2-xls-r-300m"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

In [ ]:
# Audio preprocessing function

def preprocess(example):

    waveform, sr = torchaudio.load(
        example["full_path"]
    )

    # Convert stereo to mono

    if waveform.shape[0] > 1:

        waveform = waveform.mean(
            dim=0,
            keepdim=True
        )

    # Resample to 16k

    if sr != 16000:

        resampler = torchaudio.transforms.Resample(
            sr,
            16000
        )

        waveform = resampler(waveform)

    # Extract features

    inputs = feature_extractor(
        waveform.squeeze().numpy(),
        sampling_rate=16000
    )

    example["input_values"] = inputs["input_values"][0]

    return example

In [ ]:
# Apply preprocessing

arabic_dataset = arabic_dataset.map(
    preprocess
)

Map:   0%|          | 0/118 [00:00<?, ? examples/s]

In [ ]:
# Split Arabic dataset into train and validation

arabic_split = arabic_dataset.train_test_split(
    test_size=0.1,   # 10% for validation
    seed=42          # Keep split consistent
)

# Training data
arabic_train = arabic_split["train"]

# Validation data
arabic_val = arabic_split["test"]

print(arabic_train)
print(arabic_val)

Dataset({
    features: ['full_path', 'labels', '__index_level_0__', 'input_values'],
    num_rows: 106
})
Dataset({
    features: ['full_path', 'labels', '__index_level_0__', 'input_values'],
    num_rows: 12
})


In [ ]:
# Load English training dataset

english_train = load_from_disk(
    "/content/drive/MyDrive/train_ready"
)

print(english_train)

Dataset({
    features: ['Show', 'EpId', 'ClipId', 'Start', 'Stop', 'Unsure', 'PoorAudioQuality', 'Prolongation', 'Block', 'SoundRep', 'WordRep', 'DifficultToUnderstand', 'Interjection', 'NoStutteredWords', 'NaturalPause', 'Music', 'NoSpeech', 'labels', 'full_path', 'exists', 'input_values'],
    num_rows: 14074
})


In [ ]:
# Combine English and Arabic datasets

combined_train = concatenate_datasets([
    english_train,
    arabic_train
])

print(combined_train)

Dataset({
    features: ['Show', 'EpId', 'ClipId', 'Start', 'Stop', 'Unsure', 'PoorAudioQuality', 'Prolongation', 'Block', 'SoundRep', 'WordRep', 'DifficultToUnderstand', 'Interjection', 'NoStutteredWords', 'NaturalPause', 'Music', 'NoSpeech', 'labels', 'full_path', 'exists', 'input_values', '__index_level_0__'],
    num_rows: 14180
})


In [ ]:
combined_train = combined_train.map(fix_labels)
arabic_val = arabic_val.map(fix_labels)

Map:   0%|          | 0/14180 [00:00<?, ? examples/s]

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForAudioClassification

# Load model for multi-label classification

model = AutoModelForAudioClassification.from_pretrained(
    "facebook/wav2vec2-xls-r-300m",
    num_labels=5
)

# Force multi-label classification
model.config.problem_type = "multi_label_classification"

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
project_hid.bias             | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
projector.bias               | MISSING    | 
classifier.bias              | MISSING    | 
classifier.weight            | MISSING    | 
projector.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
print(model.config.problem_type)

multi_label_classification


In [ ]:
# Metrics for multi-label classification

from sklearn.metrics import f1_score
import numpy as np

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    # Convert logits to probabilities
    probs = 1 / (1 + np.exp(-logits))

    # Convert probabilities to 0 or 1
    preds = (probs > 0.5).astype(int)

    # Calculate macro F1 score
    f1 = f1_score(
        labels,
        preds,
        average="macro"
    )

    return {
        "f1": f1
    }

In [ ]:
# Create batches for training

def collate_fn(batch):

    # Convert input_values to tensors
    input_values = [
        torch.tensor(x["input_values"])
        for x in batch
    ]

    # Pad audio to same length
    input_values = torch.nn.utils.rnn.pad_sequence(
        input_values,
        batch_first=True
    )

    # Convert labels to float tensor
    labels = torch.tensor(
        [x["labels"] for x in batch],
        dtype=torch.float32
    )

    return {
        "input_values": input_values,
        "labels": labels
    }

In [ ]:
print(combined_train[0]["labels"])
print(type(combined_train[0]["labels"]))

[1.0, 0.0, 0.0, 1.0, 1.0]
<class 'list'>


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(

    output_dir="/content/drive/MyDrive/stuttering_model",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=5e-6,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=8,

    num_train_epochs=8,

    warmup_ratio=0.1,

    weight_decay=0.01,

    fp16=True,

    load_best_model_at_end=True,

    metric_for_best_model="f1",
    greater_is_better=True,

    save_total_limit=2,

    logging_steps=10,

    report_to="none"
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
from transformers import Trainer
import torch.nn as nn
class MultiLabelTrainer(Trainer):

    def compute_loss(

        self,

        model,

        inputs,

        return_outputs=False,

        num_items_in_batch=None

    ):

        labels = inputs.get("labels")

        outputs = model(

            input_values=inputs.get("input_values")

        )

        logits = outputs.get("logits")

        loss_fct = torch.nn.BCEWithLogitsLoss()

        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

trainer = MultiLabelTrainer(

    model=model,

    args=training_args,

    train_dataset=combined_train,

    eval_dataset=arabic_val,

    compute_metrics=compute_metrics,

    data_collator=collate_fn

)


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,2.614174,0.379651,0.521434
2,2.248392,0.386835,0.534852
3,2.700112,0.376737,0.510169
4,2.366707,0.368893,0.532271
5,2.415280,0.372747,0.514058
6,2.623659,0.374132,0.530198
7,2.568658,0.375216,0.518502
8,2.276624,0.375023,0.518502


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3608, training_loss=2.4482911003138166, metrics={'train_runtime': 6160.7889, 'train_samples_per_second': 18.727, 'train_steps_per_second': 0.586, 'total_flos': 1.0516805632301974e+19, 'train_loss': 2.4482911003138166, 'epoch': 8.0})

In [ ]:
preds = trainer.predict(arabic_val)

logits = preds.predictions
labels = preds.label_ids

probs = 1 / (1 + np.exp(-logits))

In [ ]:
from sklearn.metrics import (
    f1_score,
    classification_report
)

import numpy as np

In [ ]:
TARGET_LABELS = [
    "Prolongation",
    "Block",
    "SoundRep",
    "WordRep",
    "Interjection"
]

best_thresholds = []

for i, name in enumerate(TARGET_LABELS):

    best_t = 0.3
    best_f1 = 0

    if name in ["Block", "Interjection"]:
        thresholds = np.arange(0.05, 0.6, 0.01)
    else:
        thresholds = np.arange(0.1, 0.8, 0.01)

    for t in thresholds:

        pred_i = (probs[:, i] > t).astype(int)

        f1 = f1_score(labels[:, i], pred_i, zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    print(f"{name} → {best_t:.2f} | F1: {best_f1:.3f}")

    best_thresholds.append(best_t)

Prolongation → 0.29 | F1: 0.737
Block → 0.33 | F1: 0.476
SoundRep → 0.34 | F1: 0.764
WordRep → 0.20 | F1: 0.625
Interjection → 0.13 | F1: 0.414


In [ ]:
final_preds = (
    probs > best_thresholds
).astype(int)


In [ ]:
print(classification_report(
    labels,
    final_preds,
    target_names=TARGET_LABELS,
    zero_division=0
))

              precision    recall  f1-score   support

Prolongation       0.78      0.70      0.74        10
       Block       0.45      0.50      0.48        20
    SoundRep       0.70      0.84      0.76        25
     WordRep       0.56      0.71      0.62         7
Interjection       0.33      0.55      0.41        11

   micro avg       0.56      0.67      0.61        73
   macro avg       0.56      0.66      0.60        73
weighted avg       0.57      0.67      0.62        73
 samples avg       0.42      0.53      0.45        73



In [ ]:
save_path = "/content/drive/MyDrive/stutter_arabic_english_v2"

trainer.save_model(save_path)

np.save(
    f"{save_path}/thresholds.npy",
    best_thresholds
)

np.save(
    f"{save_path}/probs.npy",
    probs
)

np.save(
    f"{save_path}/labels.npy",
    labels
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
#Validation the model on Arabic dataset

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import torchaudio

from datasets import Dataset
from transformers import (
    AutoFeatureExtractor,
    AutoModelForAudioClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import f1_score, classification_report

In [ ]:
# Load Arabic Excel file
excel_path = "/content/drive/MyDrive/تأتأة .xlsx"

df_raw = pd.read_excel(excel_path, sheet_name=1, header=None)

# Skip header rows (your data starts from row 3)
df = df_raw.iloc[3:].reset_index(drop=True)

# Keep only needed columns
df = df.iloc[:, [2, 3]]
df.columns = ["ClipId", "label_text"]

# Remove empty rows
df = df.dropna()

# Convert ClipId to numeric
df["ClipId"] = pd.to_numeric(df["ClipId"], errors="coerce")
df = df.dropna(subset=["ClipId"])
df["ClipId"] = df["ClipId"].astype(int)
df["ClipId_str"] = df["ClipId"].astype(str)

print(df.head())

   ClipId        label_text ClipId_str
0       2  NoStutteredWords          2
1       3   Block, SoundRep          3
2       4   Block, SoundRep          4
3       5            Block           5
4       6  NoStutteredWords          6


In [ ]:
# Map audio files to ClipId
audio_folder = "/content/drive/MyDrive/Audio_All"

audio_map = {}
for f in os.listdir(audio_folder):
    name = f.replace(".wav", "")
    if name.isdigit():
        audio_map[name] = os.path.join(audio_folder, f)

# Add full audio path to dataframe
df["audio_path"] = df["ClipId_str"].map(audio_map)

# Remove rows without audio
df = df.dropna(subset=["audio_path"])

print(df.head())

   ClipId        label_text ClipId_str                              audio_path
0       2  NoStutteredWords          2  /content/drive/MyDrive/Audio_All/2.wav
1       3   Block, SoundRep          3  /content/drive/MyDrive/Audio_All/3.wav
2       4   Block, SoundRep          4  /content/drive/MyDrive/Audio_All/4.wav
3       5            Block           5  /content/drive/MyDrive/Audio_All/5.wav
4       6  NoStutteredWords          6  /content/drive/MyDrive/Audio_All/6.wav


In [ ]:
# Define target labels
TARGET_LABELS = [
    "Prolongation",
    "Block",
    "SoundRep",
    "WordRep",
    "Interjection"
]

# Convert text labels to multi-hot vector
def encode_labels(row):
    return np.array([
        int(label.lower() in str(row["label_text"]).lower())
        for label in TARGET_LABELS
    ], dtype=np.float32)

df["labels"] = df.apply(encode_labels, axis=1)

In [ ]:
# Create dataset format for HuggingFace
df["full_path"] = df["audio_path"]

dataset = Dataset.from_pandas(
    df[["full_path", "labels"]]
)

print(dataset)

Dataset({
    features: ['full_path', 'labels', '__index_level_0__'],
    num_rows: 436
})


In [ ]:
# Load wav2vec2 feature extractor
feature_extractor = AutoFeatureExtractor.from_pretrained(
    "facebook/wav2vec2-xls-r-300m"
)

In [ ]:
def preprocess(example):
    waveform, sr = torchaudio.load(example["full_path"])

    # mono (convert stereo to mono)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    # resample to 16kHz
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(sr, 16000)
        waveform = resampler(waveform)

    # convert to 1D numpy array
    example["input_values"] = waveform.squeeze().numpy()

    return example

dataset = dataset.map(preprocess)

Map:   0%|          | 0/436 [00:00<?, ? examples/s]

In [ ]:
# Split dataset into training and validation
split = dataset.train_test_split(test_size=0.2, seed=42)

train_dataset = split["train"]
val_dataset = split["test"]

In [ ]:
# Ensure labels are float tensor format
def fix_labels(example):
    example["labels"] = torch.tensor(
        example["labels"],
        dtype=torch.float32
    ).tolist()
    return example

train_dataset = train_dataset.map(fix_labels)
val_dataset = val_dataset.map(fix_labels)

Map:   0%|          | 0/348 [00:00<?, ? examples/s]

Map:   0%|          | 0/88 [00:00<?, ? examples/s]

In [ ]:
# Load pretrained wav2vec2 model
model = AutoModelForAudioClassification.from_pretrained(
    "facebook/wav2vec2-xls-r-300m",
    num_labels=5,
    problem_type="multi_label_classification"
)

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
quantizer.codevectors        | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
classifier.bias              | MISSING    | 
projector.weight             | MISSING    | 
projector.bias               | MISSING    | 
classifier.weight            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import torch
from transformers import DataCollatorWithPadding

def collate_fn(batch):
    input_values = [torch.tensor(x["input_values"], dtype=torch.float32) for x in batch]

    # pad audio to same length
    input_values = torch.nn.utils.rnn.pad_sequence(
        input_values,
        batch_first=True
    )

    labels = torch.tensor(
        [x["labels"] for x in batch],
        dtype=torch.float32
    )

    return {
        "input_values": input_values,
        "labels": labels
    }

In [ ]:
# Training configuration
training_args = TrainingArguments(
    output_dir="/content/stutter_arabic_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=5,
    logging_steps=10,
    report_to="none"
)

In [ ]:
from transformers import Trainer
import torch
import torch.nn as nn

class MultiLabelTrainer(Trainer):

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):

        # Get labels
        labels = inputs["labels"].float()

        # Forward pass
        outputs = model(
            input_values=inputs["input_values"]
        )

        # Get logits
        logits = outputs.logits

        # Loss function for multi-label
        loss_fct = torch.nn.BCEWithLogitsLoss()

        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss


trainer = MultiLabelTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.472588,0.456081
2,0.472918,0.438082
3,0.454695,0.433176
4,0.451114,0.432051
5,0.418446,0.431688


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=435, training_loss=0.45069629899386704, metrics={'train_runtime': 134.6965, 'train_samples_per_second': 12.918, 'train_steps_per_second': 3.229, 'total_flos': 1.4656048482806102e+17, 'train_loss': 0.45069629899386704, 'epoch': 5.0})

In [ ]:
preds = trainer.predict(val_dataset)

logits = preds.predictions
labels = preds.label_ids

In [ ]:
import torch
import numpy as np

probs = torch.sigmoid(torch.tensor(logits)).numpy()

In [ ]:
thresholds = np.array([0.25, 0.20, 0.40, 0.30, 0.30])

predictions = (probs > thresholds).astype(int)

In [ ]:
from sklearn.metrics import classification_report

TARGET_LABELS = [
    "Prolongation",
    "Block",
    "SoundRep",
    "WordRep",
    "Interjection"
]

print(classification_report(labels, predictions, target_names=TARGET_LABELS))

              precision    recall  f1-score   support

Prolongation       0.00      0.00      0.00        10
       Block       0.23      1.00      0.37        20
    SoundRep       0.00      0.00      0.00        25
     WordRep       0.00      0.00      0.00         7
Interjection       0.00      0.00      0.00        11

   micro avg       0.23      0.27      0.25        73
   macro avg       0.05      0.20      0.07        73
weighted avg       0.06      0.27      0.10        73
 samples avg       0.23      0.21      0.22        73



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from sklearn.metrics import f1_score

best_thresholds = []

for i, name in enumerate(TARGET_LABELS):
    best_t = 0.5
    best_f1 = 0

    for t in np.arange(0.1, 0.9, 0.05):
        pred_i = (probs[:, i] > t).astype(int)
        f1 = f1_score(labels[:, i], pred_i)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    print(f"{name} → best threshold: {best_t:.2f} | F1: {best_f1:.3f}")
    best_thresholds.append(best_t)

best_thresholds = np.array(best_thresholds)

Prolongation → best threshold: 0.10 | F1: 0.204
Block → best threshold: 0.10 | F1: 0.370
SoundRep → best threshold: 0.10 | F1: 0.442
WordRep → best threshold: 0.50 | F1: 0.000
Interjection → best threshold: 0.10 | F1: 0.222


In [ ]:
final_predictions = (probs > best_thresholds).astype(int)

In [ ]:
from sklearn.metrics import classification_report

TARGET_LABELS = [
    "Prolongation",
    "Block",
    "SoundRep",
    "WordRep",
    "Interjection"
]

print(classification_report(
    labels,
    final_predictions,
    target_names=TARGET_LABELS
))

              precision    recall  f1-score   support

Prolongation       0.11      1.00      0.20        10
       Block       0.23      1.00      0.37        20
    SoundRep       0.28      1.00      0.44        25
     WordRep       0.00      0.00      0.00         7
Interjection       0.12      1.00      0.22        11

   micro avg       0.19      0.90      0.31        73
   macro avg       0.15      0.80      0.25        73
weighted avg       0.19      0.90      0.31        73
 samples avg       0.19      0.70      0.29        73



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
#create Dataset

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torchaudio
from datasets import Dataset

In [ ]:
excel_path = "/content/drive/MyDrive/تأتأة .xlsx"

df_raw = pd.read_excel(excel_path, sheet_name=1, header=None)

df = df_raw.iloc[3:].reset_index(drop=True)

In [ ]:
df = df.iloc[:, [2, 3]]
df.columns = ["ClipId", "label_text"]

df = df.dropna()

df["ClipId"] = pd.to_numeric(df["ClipId"], errors="coerce")
df = df.dropna(subset=["ClipId"])
df["ClipId"] = df["ClipId"].astype(int)
df["ClipId_str"] = df["ClipId"].astype(str)

In [ ]:
audio_folder = "/content/drive/MyDrive/Audio_All"

audio_map = {}
for f in os.listdir(audio_folder):
    name = f.replace(".wav", "")
    if name.isdigit():
        audio_map[name] = os.path.join(audio_folder, f)

df["audio_path"] = df["ClipId_str"].map(audio_map)
df = df.dropna(subset=["audio_path"])

In [ ]:
TARGET_LABELS = [
    "Prolongation",
    "Block",
    "SoundRep",
    "WordRep",
    "Interjection"
]

def encode_labels(row):
    return np.array([
        int(label.lower() in str(row["label_text"]).lower())
        for label in TARGET_LABELS
    ], dtype=np.float32)

df["labels"] = df.apply(encode_labels, axis=1)
df["full_path"] = df["audio_path"]

In [ ]:
dataset = Dataset.from_pandas(
    df[["full_path", "labels"]]
)

print(dataset)

Dataset({
    features: ['full_path', 'labels', '__index_level_0__'],
    num_rows: 436
})


In [ ]:
def preprocess(example):
    waveform, sr = torchaudio.load(example["full_path"])

    # mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    # resample
    if sr != 16000:
        resampler = torchaudio.transforms.Resample(sr, 16000)
        waveform = resampler(waveform)

    example["input_values"] = waveform.squeeze().numpy()
    return example

dataset = dataset.map(preprocess)

Map:   0%|          | 0/436 [00:00<?, ? examples/s]

In [ ]:
split = dataset.train_test_split(test_size=0.2, seed=42)

train_dataset = split["train"]
val_dataset = split["test"]

In [ ]:
train_dataset.save_to_disk("/content/drive/MyDrive/arabic_train_ready")
val_dataset.save_to_disk("/content/drive/MyDrive/arabic_val_ready")

Saving the dataset (0/1 shards):   0%|          | 0/348 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/88 [00:00<?, ? examples/s]

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import torch.nn.utils.rnn as rnn_utils
from transformers import Wav2Vec2Model, Trainer, TrainingArguments
from sklearn.metrics import classification_report, f1_score

In [ ]:
train_dataset.save_to_disk("/content/drive/MyDrive/arabic_train_ready")
val_dataset.save_to_disk("/content/drive/MyDrive/arabic_val_ready")

Saving the dataset (0/1 shards):   0%|          | 0/615 [00:00<?, ? examples/s]

PermissionError: Tried to overwrite /content/drive/MyDrive/arabic_val_ready but a dataset can't overwrite itself.

In [ ]:
labels_array = np.array(train_dataset["labels"])
class_counts = labels_array.sum(axis=0)

weights = class_counts.max() / (class_counts + 1e-6)
weights = torch.tensor(weights, dtype=torch.float32)

In [ ]:
base = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-xls-r-300m")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.bias             | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
class StutterModel(nn.Module):
    def __init__(self, base, weights):
        super().__init__()
        self.base = base
        hidden = base.config.hidden_size

        # better attention (slightly deeper)
        self.attention = nn.Sequential(
            nn.Linear(hidden, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        # stronger classifier (helps F1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 5)
        )

        self.weights = weights

    def forward(self, input_values, labels=None):
        outputs = self.base(input_values)
        x = outputs.last_hidden_state

        attn = torch.softmax(self.attention(x), dim=1)
        x = (x * attn).sum(dim=1)

        logits = self.classifier(x)

        loss = None
        if labels is not None:
            loss_fn = nn.BCEWithLogitsLoss(
                pos_weight=self.weights.to(logits.device)
            )
            loss = loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}

In [ ]:
def collate_fn(batch):
    input_values = [
        torch.tensor(x["input_values"], dtype=torch.float32)
        for x in batch
    ]

    input_values = rnn_utils.pad_sequence(input_values, batch_first=True)

    labels = torch.tensor(
        np.array([x["labels"] for x in batch]),
        dtype=torch.float32
    )

    return {
        "input_values": input_values,
        "labels": labels
    }

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./stutter_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=8,
    weight_decay=0.01,
    logging_steps=10,
    report_to="none"
)

In [ ]:
model = StutterModel(base, weights)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn
)


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.720849,0.686640
2,0.741211,0.692282
3,0.687005,0.694947
4,0.629501,0.699081
5,0.602179,0.692236
6,0.562820,0.680821
7,0.589573,0.673448
8,0.665698,0.673298


TrainOutput(global_step=1232, training_loss=0.6530209218526816, metrics={'train_runtime': 258.6665, 'train_samples_per_second': 19.021, 'train_steps_per_second': 4.763, 'total_flos': 0.0, 'train_loss': 0.6530209218526816, 'epoch': 8.0})

In [ ]:
preds = trainer.predict(val_dataset)

logits = preds.predictions
labels = preds.label_ids

probs = torch.sigmoid(torch.tensor(logits)).numpy()

In [ ]:
TARGET_LABELS = [
    "Prolongation",
    "Block",
    "SoundRep",
    "WordRep",
    "Interjection"
]

best_thresholds = []

for i, name in enumerate(TARGET_LABELS):
    best_t = 0.1
    best_f1 = 0

    for t in np.arange(0.05, 0.5, 0.02):
        pred_i = (probs[:, i] > t).astype(int)
        f1 = f1_score(labels[:, i], pred_i, zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    print(f"{name} → best threshold: {best_t:.2f} | F1: {best_f1:.3f}")
    best_thresholds.append(best_t)

best_thresholds = np.array(best_thresholds)

Prolongation → best threshold: 0.47 | F1: 0.391
Block → best threshold: 0.15 | F1: 0.374
SoundRep → best threshold: 0.15 | F1: 0.451
WordRep → best threshold: 0.27 | F1: 0.667
Interjection → best threshold: 0.35 | F1: 0.278


In [ ]:
final_predictions = (probs > best_thresholds).astype(int)

print(classification_report(
    labels,
    final_predictions,
    target_names=TARGET_LABELS,
    zero_division=0
))

              precision    recall  f1-score   support

Prolongation       0.25      0.90      0.39        10
       Block       0.23      1.00      0.37        20
    SoundRep       0.30      0.92      0.45        25
     WordRep       0.80      0.57      0.67         7
Interjection       0.16      1.00      0.28        11

   micro avg       0.25      0.92      0.39        73
   macro avg       0.35      0.88      0.43        73
weighted avg       0.30      0.92      0.42        73
 samples avg       0.26      0.72      0.37        73



In [ ]:
import numpy as np

trainer.save_model("/content/drive/MyDrive/arabic_stutter_model_v6")

np.save("/content/drive/MyDrive/arabic_v6_best_thresholds.npy", best_thresholds)
np.save("/content/drive/MyDrive/arabic_v6_val_probs.npy", probs)
np.save("/content/drive/MyDrive/arabic_v6_val_labels.npy", labels)

trainer.save_state()

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
train_dataset.save_to_disk("/content/drive/MyDrive/arabic_train_ready")
val_dataset.save_to_disk("/content/drive/MyDrive/arabic_val_ready")

Saving the dataset (0/1 shards):   0%|          | 0/348 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/88 [00:00<?, ? examples/s]

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.utils.rnn as rnn_utils
from transformers import Wav2Vec2Model, Trainer, TrainingArguments
from datasets import load_from_disk
from sklearn.metrics import classification_report, f1_score

In [ ]:
TARGET_LABELS = [
    "Prolongation",
    "Block",
    "SoundRep",
    "WordRep",
    "Interjection"
]

labels_array = np.array(train_dataset["labels"])
class_counts = labels_array.sum(axis=0)

weights = class_counts.max() / (class_counts + 1e-6)
weights = torch.tensor(weights, dtype=torch.float32)

In [ ]:
base = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-xls-r-300m")

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
class StutterModel(nn.Module):
    def __init__(self, base, weights):
        super().__init__()
        self.base = base
        hidden = base.config.hidden_size

        self.attention = nn.Sequential(
            nn.Linear(hidden, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 5)
        )

        self.weights = weights

    def forward(self, input_values, labels=None):
        x = self.base(input_values).last_hidden_state

        attn = torch.softmax(self.attention(x), dim=1)
        x = (x * attn).sum(dim=1)

        logits = self.classifier(x)

        loss = None
        if labels is not None:
            loss_fn = nn.BCEWithLogitsLoss(
                pos_weight=self.weights.to(logits.device)
            )
            loss = loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}

In [ ]:
def collate_fn(batch):
    input_values = [
        torch.tensor(x["input_values"], dtype=torch.float32)
        for x in batch
    ]

    input_values = rnn_utils.pad_sequence(input_values, batch_first=True)

    labels = torch.tensor(
        np.array([x["labels"] for x in batch]),
        dtype=torch.float32
    )

    return {
        "input_values": input_values,
        "labels": labels
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./stutter_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=8,
    weight_decay=0.01,
    logging_steps=10,
    report_to="none"
)

In [ ]:
model = StutterModel(base, weights)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.690268,0.687826
2,0.741612,0.663996
3,0.684817,0.658047
4,0.745639,0.658601
5,0.600525,0.658809
6,0.683317,0.659996
7,0.633478,0.660831
8,0.601666,0.660839


TrainOutput(global_step=696, training_loss=0.6516922314961752, metrics={'train_runtime': 195.2373, 'train_samples_per_second': 14.26, 'train_steps_per_second': 3.565, 'total_flos': 0.0, 'train_loss': 0.6516922314961752, 'epoch': 8.0})

In [ ]:
preds = trainer.predict(val_dataset)

logits = preds.predictions
labels = preds.label_ids

probs = torch.sigmoid(torch.tensor(logits)).numpy()

In [ ]:
custom_thresholds = np.array([
    0.40,  # Prolongation
    0.60,  # Block
    0.55,  # SoundRep
    0.45,  # WordRep
    0.65   # Interjection
])

In [ ]:
final_predictions = (probs > best_thresholds).astype(int)

In [ ]:
print(classification_report(
    labels,
    final_predictions,
    target_names=TARGET_LABELS,
    zero_division=0
))

              precision    recall  f1-score   support

Prolongation       0.27      0.40      0.32        10
       Block       0.23      1.00      0.37        20
    SoundRep       0.28      1.00      0.44        25
     WordRep       0.60      0.43      0.50         7
Interjection       0.13      1.00      0.23        11

   micro avg       0.22      0.86      0.35        73
   macro avg       0.30      0.77      0.37        73
weighted avg       0.27      0.86      0.38        73
 samples avg       0.23      0.67      0.33        73



In [ ]:
#Connect new model to app

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import torch.nn as nn
import numpy as np
from transformers import Wav2Vec2Model

In [ ]:
class StutterModelV4(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        hidden = base.config.hidden_size

        self.attention = nn.Linear(hidden, 1)

        self.classifier = nn.Sequential(
            nn.Linear(hidden, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 5)
        )

    def forward(self, input_values):
        outputs = self.base(input_values)
        x = outputs.last_hidden_state

        attn = torch.softmax(self.attention(x), dim=1)
        x = (x * attn).sum(dim=1)

        logits = self.classifier(x)
        return logits

In [ ]:
base = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-xls-r-300m")

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from safetensors.torch import load_file

MODEL_DIR = "/content/drive/MyDrive/stutter_arabic_english_v2"

state_dict = load_file(f"{MODEL_DIR}/model.safetensors")

In [ ]:
from transformers import AutoModelForAudioClassification
from safetensors.torch import load_file
import numpy as np

MODEL_DIR = "/content/drive/MyDrive/stutter_arabic_english_v2"

model = AutoModelForAudioClassification.from_pretrained(MODEL_DIR)
model.eval()

thresholds = np.load(f"{MODEL_DIR}/thresholds.npy")
LABELS = np.load(f"{MODEL_DIR}/labels.npy", allow_pickle=True).tolist()

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]